In [2]:
import pandas as pd

In [3]:
fake_df = pd.read_csv(r"C:\Users\iamsi\Basic-Project2-Fake_News_Detection\data\Fake.csv")
true_df = pd.read_csv(r"C:\Users\iamsi\Basic-Project2-Fake_News_Detection\data\True.csv")

In [4]:
fake_df.head()


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [5]:
true_df.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [6]:
print(fake_df.shape)

print(true_df.shape)

(23481, 4)
(21417, 4)


In [7]:
fake_df["label"] = 0

In [8]:
true_df["label"] = 1

In [9]:
fake_df.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [10]:
df = pd.concat([fake_df, true_df], ignore_index=True)

In [11]:
df

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0
...,...,...,...,...,...
44893,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",1
44894,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",1
44895,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",1
44896,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",1


In [12]:
df = df.sample(frac=1, random_state=42)

In [13]:
df = df.reset_index(drop=True)

In [14]:
df.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [15]:
df["label"].value_counts()

label
0    23481
1    21417
Name: count, dtype: int64

In [16]:
df["content"] = df["title"] + " " + df["text"]

In [17]:
df[["content", "label"]].head()

,content,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,Trump drops Steve Bannon from National Securit...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,0
4,Donald Trump heads for Scotland to reopen a go...,1


## Cleaning Text

In [18]:
import re

def clean_text(text):
    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [19]:
df["content"] = df["content"].apply(clean_text)

In [20]:
df["content"].head()

0    ben stein calls out th circuit court committed...
1    trump drops steve bannon from national securit...
2    puerto rico expects u s to lift jones act ship...
3    oops trump just accidentally confirmed he leak...
4    donald trump heads for scotland to reopen a go...
Name: content, dtype: object

In [21]:
X = df["content"]
y = df["label"]

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1, 2),  
    min_df=2,            
    max_df=0.9           
)

In [24]:
X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

In [25]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(35918, 10000)
(8980, 10000)


## Model Training

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

In [27]:
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight='balanced', C=1.0 ),
}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)

    y_pred = model.predict(X_test_tfidf)

    print(f"{name}: {accuracy_score(y_test, y_pred):.4f}")

Logistic Regression: 0.9888
Naive Bayes: 0.9477
Linear SVM: 0.9950


In [28]:
models.keys()

dict_keys(['Logistic Regression', 'Naive Bayes', 'Linear SVM'])

In [29]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "Linear SVM",
    ],
    "Accuracy": [
        0.9860,
        0.9308,
        0.9945,
    ]
})

results.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy
2,Linear SVM,0.9945
0,Logistic Regression,0.9860
1,Naive Bayes,0.9308


### Final Model Selection

Linear SVM was selected as the final model because it achieved the highest accuracy (**99.45%**) among the tested linear models and is well-suited for text classification tasks using TF-IDF features.


In [30]:
import joblib

In [31]:
svm_model = models["Linear SVM"]

In [32]:
joblib.dump(svm_model, r"C:\Users\iamsi\Basic-Project2-Fake_News_Detection\models\linear_svm_model.pkl")
joblib.dump(vectorizer, r"C:\Users\iamsi\Basic-Project2-Fake_News_Detection\models\tfidf_vectorizer.pkl")

['C:\\Users\\iamsi\\Basic-Project2-Fake_News_Detection\\models\\tfidf_vectorizer.pkl']

In [35]:
df['content'][10]

'new jersey s christie mulls run to lead republican party report washington reuters new jersey governor chris christie an early supporter of u s president elect donald trump is considering a run to lead the republican national committee politico reported on thursday reuters could not immediately confirm the report and representatives for christie did not immediately respond to a request for comment christie whose term as governor ends in january had been leading trump s transition team until u s vice president elect mike pence recently took over earlier this month current rnc chairman reince priebus has been tapped to serve as trump s chief of staff when he starts his white house term jan christie had launched a presidential bid alongside trump in a pool of candidates that eventually saw trump a new york businessman who had never held political office win the nomination as well as the nov presidential election over democrat hillary clinton christie had been a rising political star befo